In [30]:
from hello.deepWorld.models.wav2vec2_hf import feature_extractor,get_wav2vec_model,preprocess_function,BASE_TRAINING_ARGUMENTS,compute_metrics
from hello.dataWorld.dataset.pangramDataLoaders import SpeakerSignalDataset
from transformers import TrainingArguments
from transformers.trainer import Trainer 

In [16]:
DATA_PATH_TRAIN="../data/data_flat/sp_audio_dataset_mq_trim25/train"
DATA_PATH_VAL="../data/data_flat/sp_audio_dataset_mq_trim25/val"
RES_PATH="../results/models_ckpt/wav2vec2"
LOG_PATH="../results/models_log/wav2vec2"

In [7]:
train=SpeakerSignalDataset(DATA_PATH_TRAIN,signalType="wav",prepro_func=preprocess_function)
val=SpeakerSignalDataset(DATA_PATH_VAL,signalType="wav",prepro_func=preprocess_function)

In [8]:

model=get_wav2vec_model(len(train.id2label),train.label2id,train.id2label)

c:\Users\emman\anaconda3\envs\datascience\lib\site-packages\transformers\configuration_utils.py:369: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Some weights of the model checkpoint at facebook/wav2vec2-base were not used when initializing Wav2Vec2ForSequenceClassification: ['quantizer.weight_proj.bias', 'quantizer.codevectors', 'project_hid.bias', 'project_q.bias', 'project_hid.weight', 'project_q.weight', 'quantizer.weight_proj.weight']
- This IS expected if you are initializing Wav2Vec2ForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing 

In [31]:
BASE_TRAINING_ARGUMENTS

{'output_dir': './models_ckpt',
 'logging_dir': './models_log',
 'overwrite_output_dir': True,
 'evaluation_strategy': 'steps',
 'eval_steps': 10,
 'save_strategy': 'epoch',
 'learning_rate': 3e-05,
 'weight_decay': 1e-05,
 'num_train_epochs': 10,
 'per_device_train_batch_size': 32,
 'per_device_eval_batch_size': 32,
 'logging_steps': 10,
 'save_total_limit': 2}

In [35]:

train_args.dataloader_num_workers

2

In [38]:
train_args=TrainingArguments(**BASE_TRAINING_ARGUMENTS)

train_args.output_dir=RES_PATH
train_args.logging_dir=LOG_PATH
train_args.per_device_eval_batch_size=16
train_args.per_device_train_batch_size=16
train_args.max_steps=100
train_args.dataloader_num_workers=2


trainer=Trainer(
    model=model,
    args=train_args,
    train_dataset=train,
    eval_dataset=val,
    compute_metrics=compute_metrics
)



PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).
max_steps is given, it will override any value given in num_train_epochs


In [39]:
trainer.train()

c:\Users\emman\anaconda3\envs\datascience\lib\site-packages\transformers\optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 341
  Num Epochs = 5
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 100
  0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [27]:
import torch
predicted=torch.argmax(model(train[0]["input_values"].unsqueeze(0)).logits.detach()).item()
train.id2label[predicted],train[0]["label"]

('alessandro0', tensor(1))